In [1]:
# Cell 1 - Imports and data load
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

df = pd.read_csv('/content/bq-results-20260323-203006-1774297820102.csv')
print(f"Shape: {df.shape}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.4f}")

Shape: (406031, 53)
Readmission rate: 0.1743


In [2]:
# Cell 2 - Preprocessing
# Fill missing values
df['marital_status'] = df['marital_status'].fillna('UNKNOWN')
df['language'] = df['language'].fillna('UNKNOWN')
df['insurance'] = df['insurance'].fillna('UNKNOWN')
df['admission_location'] = df['admission_location'].fillna('UNKNOWN')
df['discharge_location'] = df['discharge_location'].fillna('UNKNOWN')
df['race'] = df['race'].fillna('UNKNOWN')

lab_cols = ['num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min',
            'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max',
            'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max']
for col in lab_cols:
    df[col] = df[col].fillna(df[col].median())

hist_cols = ['days_since_last_discharge', 'num_admissions_last_30d',
             'num_admissions_last_90d', 'num_admissions_last_year',
             'total_prior_admissions', 'recent_admission_flag',
             'frequent_flyer_flag']
for col in hist_cols:
    df[col] = df[col].fillna(0)

df = df.fillna(df.median(numeric_only=True))

# Encode categoricals
categorical_cols = ['gender', 'race', 'marital_status', 'language', 'insurance',
                    'admission_location', 'discharge_location', 'admission_type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Define features
drop_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'readmitted_30d', 'readmitted_60d', 'readmitted_90d',
             'days_to_next_admission']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df['readmitted_30d']

print(f"Features: {len(feature_cols)}")
print(f"Class distribution: {y.value_counts().to_dict()}")

Features: 45
Class distribution: {0: 335240, 1: 70791}


In [3]:
import pickle

feature_cols = pickle.load(open('/content/final_feature_cols.pkl', 'rb'))
split = pickle.load(open('/content/final_split_indices.pkl', 'rb'))
train_end = split['train_end']
val_end   = split['val_end']

df['admittime'] = pd.to_datetime(df['admittime'])
df_sorted = df.sort_values('admittime').reset_index(drop=True)

X = df_sorted[feature_cols]
y = df_sorted['readmitted_30d']

X_train, y_train = X.iloc[:train_end],        y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],          y.iloc[val_end:]

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Train readmission rate: {y_train.mean():.4f}")
print(f"Test readmission rate:  {y_test.mean():.4f}")

Train: 284,221 | Val: 60,905 | Test: 60,905
Train readmission rate: 0.1734
Test readmission rate:  0.1800


In [4]:
# Cell 4 - LightGBM baseline
import lightgbm as lgb

# Class weight ratio
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Baseline model
lgb_base = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)

y_pred = lgb_base.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, y_pred)
print(f"\nLightGBM Baseline AUROC: {auroc:.4f}")

Scale pos weight: 4.77

LightGBM Baseline AUROC: 0.6871


In [5]:
# Cell 5 - LightGBM tuning (Colab optimised)
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6, 8],
    'num_leaves': [31, 63],
    'min_child_samples': [20, 50],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [0, 0.1]
}

lgb_tuned = lgb.LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1,
    verbose=-1
)

search = RandomizedSearchCV(
    lgb_tuned,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2
)

search.fit(X_train, y_train)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV AUROC: {search.best_score_:.4f}")

y_pred_tuned = search.best_estimator_.predict_proba(X_test)[:, 1]
auroc_tuned = roc_auc_score(y_test, y_pred_tuned)
print(f"LightGBM Tuned Test AUROC: {auroc_tuned:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, min_child_samples=50, n_estimators=500, num_leaves=63, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8; total time=  27.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, min_child_samples=50, n_estimators=500, num_leaves=63, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8; total time=  23.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, min_child_samples=50, n_estimators=500, num_leaves=63, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8; total time=  23.6s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_samples=50, n_estimators=700, num_leaves=31, reg_alpha=0, reg_lambda=0.1, subsample=0.8; total time=  29.0s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=6, min_child_samples=50, n_estimators=700, num_leaves=31, reg_alpha=0, reg_lambda=0.1, subsample=0.8; total time=  27.6s
[CV] END colsample_bytr

In [6]:
# Cell 6 - Save model and final comparison
import pickle

with open('/content/lgbm_tuned.pkl', 'wb') as f:
    pickle.dump(search.best_estimator_, f)
print("Saved: lgbm_tuned.pkl")

print(f"\n--- Final Model Comparison ---")
print(f"Logistic Regression:       0.7021")
print(f"LR + Class Weights:        0.7025")
print(f"LR + SMOTE:                0.6984")
print(f"XGBoost (tuned):           0.7197")
print(f"LightGBM (tuned):          {auroc_tuned:.4f}")
print(f"LSTM with race:            0.6987")
print(f"LSTM without race:         0.6998")
print(f"Target:                    0.7500")

Saved: lgbm_tuned.pkl

--- Final Model Comparison ---
Logistic Regression:       0.7021
LR + Class Weights:        0.7025
LR + SMOTE:                0.6984
XGBoost (tuned):           0.7197
LightGBM (tuned):          0.7179
LSTM with race:            0.6987
LSTM without race:         0.6998
Target:                    0.7500
